In [8]:
import pandas as pd

import SimpleITK as sitk 
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path 
from scipy.ndimage import label, find_objects

datasets = {
    "internal_train": ("/projects/vig/Datasets/aneurysm/cta_datasets/internal_train/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/internal_train/crop_0.4_label"),
    "internal_test": ("/projects/vig/Datasets/aneurysm/cta_datasets/internal_test/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/internal_test/crop_0.4_label"),
    "external": ("/projects/vig/Datasets/aneurysm/cta_datasets/external/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/external/crop_0.4_label"),
    "cmha": ("/projects/vig/Datasets/aneurysm/cta_datasets/cmha/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/cmha/crop_0.4_label"),
    "hospital140" : ("/projects/vig/Datasets/aneurysm/cta_datasets/hospital140/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/hospital140/crop_0.4_label"),
}

# for each dataset, identify each lesion in the label file, then, for each lesion compute the overlap with the vessel file and get the percentage of overlap with artery (label 1) and vein (label 2)


stats = {
    "internal_train": [],
    "internal_test": [],
    "external": [],
    "cmha": [],
    "hospital140": [],
}

for dataset_name, (vessel_path, label_path) in datasets.items():
    vessel_files = sorted(Path(vessel_path).glob("*.nii.gz"))
    label_files = sorted(Path(label_path).glob("*.nii.gz"))
    
    for vessel_file, label_file in zip(vessel_files, label_files):
        vessel_img = sitk.ReadImage(str(vessel_file))
        vessel_array = sitk.GetArrayFromImage(vessel_img)
        
        label_img = sitk.ReadImage(str(label_file))
        label_array = sitk.GetArrayFromImage(label_img)
        
        # identify lesions in label_array (connected components of label 3)
        lesion_mask = (label_array == 1).astype(np.uint8)
        labeled_lesions, num_lesions = label(lesion_mask)
        
        for lesion_id in range(1, num_lesions + 1):
            lesion_coords = np.where(labeled_lesions == lesion_id)
            lesion_size = len(lesion_coords[0])
            
            # get vessel overlap
            vessel_overlap = vessel_array[lesion_coords]
            artery_overlap = np.sum(vessel_overlap == 1)
            vein_overlap = np.sum(vessel_overlap == 2)
            
            artery_percentage = (artery_overlap / lesion_size) * 100
            vein_percentage = (vein_overlap / lesion_size) * 100
            
            stats[dataset_name].append({
                "file": str(label_file.name),
                "lesion_id": lesion_id,
                "lesion_size": lesion_size,
                "artery_percentage": artery_percentage,
                "vein_percentage": vein_percentage,
            }) 



In [7]:
stats["cmha"]

[]

In [12]:
# for each dataset, print the count of lesions out of total with either artery_percentage > 50% or vein_percentage > 50%, also print the standalone number for artery and vein separately

for dataset_name, lesion_stats in stats.items():
    total_lesions = len(lesion_stats)
    artery_dominant = sum(1 for stat in lesion_stats if stat["artery_percentage"] > 10)
    vein_dominant = sum(1 for stat in lesion_stats if stat["vein_percentage"] > 10)
    either_dominant = sum(1 for stat in lesion_stats if stat["artery_percentage"] > 10 or stat["vein_percentage"] > 10)
    
    print(f"Dataset: {dataset_name}")
    print(f"Total lesions: {total_lesions}")
    print(f"Artery intersect (>10%): {artery_dominant} ({(artery_dominant / total_lesions * 100):.2f}%)")
    print(f"Vein intersect (>10%): {vein_dominant} ({(vein_dominant / total_lesions * 100):.2f}%)")
    print(f"Either (>10%): {either_dominant} ({(either_dominant / total_lesions * 100):.2f}%)")
    print("--------------------------------------------------")


Dataset: internal_train
Total lesions: 1382
Artery intersect (>10%): 1300 (94.07%)
Vein intersect (>10%): 115 (8.32%)
Either (>10%): 1323 (95.73%)
--------------------------------------------------
Dataset: internal_test
Total lesions: 126
Artery intersect (>10%): 122 (96.83%)
Vein intersect (>10%): 6 (4.76%)
Either (>10%): 123 (97.62%)
--------------------------------------------------
Dataset: external
Total lesions: 101
Artery intersect (>10%): 94 (93.07%)
Vein intersect (>10%): 8 (7.92%)
Either (>10%): 95 (94.06%)
--------------------------------------------------
Dataset: cmha
Total lesions: 113
Artery intersect (>10%): 100 (88.50%)
Vein intersect (>10%): 10 (8.85%)
Either (>10%): 103 (91.15%)
--------------------------------------------------
Dataset: hospital140
Total lesions: 212
Artery intersect (>10%): 177 (83.49%)
Vein intersect (>10%): 19 (8.96%)
Either (>10%): 184 (86.79%)
--------------------------------------------------


In [13]:
import pandas as pd

import SimpleITK as sitk 
import numpy as np 
import matplotlib.pyplot as plt
from pathlib import Path 
from scipy.ndimage import label, find_objects

datasets = {
    "internal_train": ("/projects/vig/Datasets/aneurysm/cta_datasets/internal_train/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/internal_train/crop_0.4_label"),
    "internal_test": ("/projects/vig/Datasets/aneurysm/cta_datasets/internal_test/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/internal_test/crop_0.4_label"),
    "external": ("/projects/vig/Datasets/aneurysm/cta_datasets/external/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/external/crop_0.4_label"),
    "cmha": ("/projects/vig/Datasets/aneurysm/cta_datasets/cmha/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/cmha/crop_0.4_label"),
    "hospital140" : ("/projects/vig/Datasets/aneurysm/cta_datasets/hospital140/crop_0.4_vessel", "/projects/vig/Datasets/aneurysm/cta_datasets/hospital140/crop_0.4_label"),
}

# for each dataset, identify each lesion in the label file, then, for each lesion compute the overlap with the vessel file and get the percentage of overlap with artery (label 1) and vein (label 2)


stats = {
    "internal_train": [],
    "internal_test": [],
    "external": [],
    "cmha": [],
    "hospital140": [],
}

for dataset_name, (vessel_path, label_path) in datasets.items():
    vessel_files = sorted(Path(vessel_path).glob("*.nii.gz"))
    label_files = sorted(Path(label_path).glob("*.nii.gz"))
    
    for vessel_file, label_file in zip(vessel_files, label_files):
        vessel_img = sitk.ReadImage(str(vessel_file))
        vessel_array = sitk.GetArrayFromImage(vessel_img)
        
        label_img = sitk.ReadImage(str(label_file))
        label_array = sitk.GetArrayFromImage(label_img)
        
        vessel_array = (vessel_array > 0).astype(np.uint8)
        artery_array = (vessel_array == 1).astype(np.uint8)
        vein_array = (vessel_array == 2).astype(np.uint8)

        # dilate all arrays by 4 voxels
        from scipy.ndimage import binary_dilation, generate_binary_structure
        struct = generate_binary_structure(3, 1)
        for _ in range(4):
            vessel_array = binary_dilation(vessel_array, structure=struct).astype(np.uint8)
            artery_array = binary_dilation(artery_array, structure=struct).astype(np.uint8)
            vein_array = binary_dilation(vein_array, structure=struct).astype(np.uint8)

        # identify lesions in label_array (connected components of label 3)
        lesion_mask = (label_array == 1).astype(np.uint8)
        labeled_lesions, num_lesions = label(lesion_mask)
        for lesion_id in range(1, num_lesions + 1):
            lesion_coords = np.where(labeled_lesions == lesion_id)
            lesion_size = len(lesion_coords[0])
            
            # get vessel overlap
            vessel_overlap = vessel_array[lesion_coords]
            artery_overlap = artery_array[lesion_coords]
            vein_overlap = vein_array[lesion_coords]
            
            vessel_percentage = (np.sum(vessel_overlap) / lesion_size) * 100
            artery_percentage = (np.sum(artery_overlap) / lesion_size) * 100
            vein_percentage = (np.sum(vein_overlap) / lesion_size) * 100
            
            stats[dataset_name].append({
                "file": str(label_file.name),
                "lesion_id": lesion_id,
                "lesion_size": lesion_size,
                "vessel_percentage": vessel_percentage,
                "artery_percentage": artery_percentage,
                "vein_percentage": vein_percentage,
            })
            
        

In [14]:

for dataset_name, lesion_stats in stats.items():
    total_lesions = len(lesion_stats)
    artery_dominant = sum(1 for stat in lesion_stats if stat["artery_percentage"] > 10)
    vein_dominant = sum(1 for stat in lesion_stats if stat["vein_percentage"] > 10)
    either_dominant = sum(1 for stat in lesion_stats if stat["artery_percentage"] > 10 or stat["vein_percentage"] > 10)
    
    print(f"Dataset: {dataset_name}")
    print(f"Total lesions: {total_lesions}")
    print(f"Artery intersect (>10%): {artery_dominant} ({(artery_dominant / total_lesions * 100):.2f}%)")
    print(f"Vein intersect (>10%): {vein_dominant} ({(vein_dominant / total_lesions * 100):.2f}%)")
    print(f"Either (>10%): {either_dominant} ({(either_dominant / total_lesions * 100):.2f}%)")
    print("--------------------------------------------------")

Dataset: internal_train
Total lesions: 1382
Artery intersect (>10%): 1352 (97.83%)
Vein intersect (>10%): 0 (0.00%)
Either (>10%): 1352 (97.83%)
--------------------------------------------------
Dataset: internal_test
Total lesions: 126
Artery intersect (>10%): 123 (97.62%)
Vein intersect (>10%): 0 (0.00%)
Either (>10%): 123 (97.62%)
--------------------------------------------------
Dataset: external
Total lesions: 101
Artery intersect (>10%): 98 (97.03%)
Vein intersect (>10%): 0 (0.00%)
Either (>10%): 98 (97.03%)
--------------------------------------------------
Dataset: cmha
Total lesions: 113
Artery intersect (>10%): 108 (95.58%)
Vein intersect (>10%): 0 (0.00%)
Either (>10%): 108 (95.58%)
--------------------------------------------------
Dataset: hospital140
Total lesions: 212
Artery intersect (>10%): 191 (90.09%)
Vein intersect (>10%): 0 (0.00%)
Either (>10%): 191 (90.09%)
--------------------------------------------------


In [15]:
# save the results to a pickle file

import pickle
with open("vessel_lesion_stats.pkl", "wb") as f:
    pickle.dump(stats, f)